In [0]:
%sql
CREATE CATALOG IF NOT EXISTS vinkoscon
MANAGED LOCATION 'abfss://processeddatabricks@strfilese.dfs.core.windows.net/vinkos_managed/';

In [0]:
%sql
-- Forzamos el uso de tu nuevo catálogo
--USE CATALOG vinkoscon;

-- Creamos el esquema
--CREATE SCHEMA IF NOT EXISTS bronze;
--CREATE SCHEMA IF NOT EXISTS silver;
--CREATE SCHEMA IF NOT EXISTS gold;


INSERT INTO vinkoscon.bronze.visitas_raw (
email
,fechaPrimeraVisita
,fechaUltimaVisita
,visitasTotales
,visitasAnioActual
,visitasMesActual
,jyv
,Badmail
,Baja
,Fecha_envio
,Fecha_open
,Opens
,Opens_virales
,Fecha_click
,Clicks
,Clicks_virales
,Links
,IPs
,Navegadores
,Plataformas
,errores
,create_date
)
SELECT 
email
,fechaPrimeraVisita
,fechaUltimaVisita
,visitasTotales
,visitasAnioActual
,visitasMesActual
,jyv
,Badmail
,Baja
,Fecha_envio
,Fecha_open
,Opens
,Opens_virales
,Fecha_click
,Clicks
,Clicks_virales
,Links
,IPs
,Navegadores
,Plataformas
,errores
,cast(create_date as string) as create_date
FROM azure_sql_vinkos_catalog.dbo.usuarios_visitas_raw where cast(create_date as date) = current_date();

In [0]:
from pyspark.sql.functions import col, when, regexp_like, lit, trim ,to_timestamp ,to_date, date_format ,current_timestamp

#limpiamos el correo, aquel que no cumpla lo desechamos. Seleccionamos el resto de los datos y tomamos los del día.
df = spark.read.table("vinkoscon.bronze.visitas_raw")
emails_valido = df.filter(col("email").rlike(r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$')).select("*").where("CAST(create_date AS DATE) = CURRENT_DATE()")
sin_duplicados= emails_valido.dropDuplicates(['email'])

#Formateamos los datos para que coincidan con el formato que queremos 
df_final = sin_duplicados.withColumn(
"fechaPrimeraVisita", 
    to_timestamp("fechaPrimeraVisita", "dd/MM/yyyy HH:mm").cast("timestamp_ntz")
).withColumn(
    "fechaUltimaVisita", 
    col("fechaUltimaVisita").cast("string")
).withColumn(
    "visitasTotales",
    col("visitasTotales").cast("int")
).withColumn(
    "visitasAnioActual",
    col("visitasAnioActual").cast("int")
).withColumn(
    "visitasMesActual",
    col("visitasMesActual").cast("int")
).withColumn(
    "Clicks",
    col("Clicks").cast("int")
).withColumn(
    "Clicks_virales",
    col("Clicks_virales").cast("int")
).withColumn(
    "Opens",
    col("Opens").cast("int")
).withColumn(
    "Opens_virales",
    col("Opens_virales").cast("int")
).withColumn(
    "errores",
    col("errores").cast("int")
).withColumn(
    "Fecha_envio",
    to_timestamp("Fecha_envio", "dd/MM/yyyy HH:mm").cast("timestamp_ntz")
).withColumn(
    "Fecha_open",
    to_timestamp("Fecha_open", "dd/MM/yyyy HH:mm").cast("timestamp_ntz")
).withColumn(
    "Fecha_click",
    to_timestamp("Fecha_click", "dd/MM/yyyy HH:mm").cast("timestamp_ntz")
).withColumn(
    "create_date",
    to_timestamp("create_date", "yyyy-MM-dd HH:mm:ss").cast("timestamp_ntz")
).withColumn(
    "jyv",
    when(col("jyv") == "S", True).when(col("jyv") == "N", False)
).withColumn(
    "Badmail",
    when(col("Badmail") == "S", True).when(col("Badmail") == "N", False)
).withColumn(
    "Baja",
    when(col("Baja") == "S", True).when(col("Baja") == "N", False)
).withColumn(
    "Update_date",
    current_timestamp()
)

#Parte donde creamos los datos que van llegando para las tablas de visitantes y estadistica (recuerda usamos los últimos ya que tomamos los current date)

df_visitante = df_final.select(col("email"),col("fechaPrimeraVisita"),col("fechaUltimaVisita"),col("visitasTotales"),col("visitasAnioActual"),col("visitasMesActual"),col("create_date"),col("Update_date"))

df_estadistica=df_final.select(col("email"),col("jyv"),col("Badmail"),col("Baja"),col("Fecha_envio"),col("Fecha_open"),col("Opens"),col("Opens_virales"),col("Fecha_click"),col("Clicks"),col("Clicks_virales"),col("Links"),col("IPs"),col("Navegadores"),col("Plataformas"),col("create_date"),col("Update_date"))

df_errores=df_final.select(col("email"),col("errores"),col("create_date"),col("Update_date"))

#Convertimos el dataset visitante en una vista temporal para poder hacer el merge con la tabla visitantes
df_visitante.createOrReplaceTempView("vista_nuevos_visitantes")

#Salvamos los datos de estadística y errores en modo append ya que no se solicita hacer merge 

#df_estadistica.write.mode("append").format("delta").saveAsTable("vinkoscon.silver.estadistica")
#df_errores.write.mode("append").format("delta").saveAsTable("vinkoscon.silver.errores") 
df_visitante.write.mode("append").format("delta").saveAsTable("vinkoscon.silver.visitas") 

#display(df_estadistica)



In [0]:
%sql
MERGE INTO vinkoscon.silver.visitas AS t
USING vista_nuevos_visitantes AS s
ON t.email = s.email
WHEN MATCHED THEN
  UPDATE SET 
    t.fechaUltimaVisita = s.fechaUltimaVisita,
    t.visitasTotales = s.visitasTotales,
    t.visitasAnioActual = s.visitasAnioActual,
    t.visitasMesActual = s.visitasMesActual,
    t.create_date = s.create_date,
    t.Update_date = current_timestamp
WHEN NOT MATCHED THEN
  INSERT (
    email, 
    fechaUltimaVisita, 
    visitasTotales, 
    visitasAnioActual, 
    visitasMesActual,
    create_date,
    Update_date
  )
  VALUES (
    s.email, 
    s.fechaUltimaVisita, 
    s.visitasTotales, 
    s.visitasAnioActual, 
    s.visitasMesActual,
    s.create_date,
    s.Update_date
  );